# Silver layer: Data cleaning and enrichment

In [0]:
# We also import the required functions for transformations, ​
# such as handling timestamps and working with datetime formats. 
from pyspark.sql.functions import col, to_timestamp, hour, minute, date_format

import pyspark.sql.functions as sf

# we ​load the bronze tables into data frames.
df_trips = spark.table("taxicatalog.taxi_project_schema.bronze_taxi_trips")
df_zones = spark.table("taxicatalog.taxi_project_schema.bronze_taxi_zones")

In [0]:
# convert the string timestamp to proper timestamp format
df_trips = df_trips.withColumn(
    "pickup_time",
    to_timestamp("tpep_pickup_datetime", "yy/MM/dd H:mm")
).withColumn(
    "dropoff_time",
    to_timestamp("tpep_dropoff_datetime", "yy/MM/dd H:mm")
)

In [0]:
# Cleaning data 
df_trips_clean = df_trips.filter(
    sf.col("pickup_time").isNotNull() &
    sf.col("dropoff_time").isNotNull() &
    sf.col("PULocationID").isNotNull() &
    sf.col("DOLocationID").isNotNull() &
    (sf.col("trip_distance")>0) &
    (sf.col("total_amount")>0)
)

In [0]:
# Extract only TIME features
df_trips_clean = df_trips_clean \
    .withColumn("pickup_hour", hour("pickup_time")) \
    .withColumn("pickup_minute", minute("pickup_time")) \
    .withColumn("dropoff_hour", hour("dropoff_time")) \
    .withColumn("pickup_time_only", date_format("pickup_time", "HH:mm:ss")) \
    .withColumn("dropoff_time_only", date_format("dropoff_time", "HH:mm:ss"))

In [0]:
df_enriched = df_trips_clean.alias("trips") \
    .join(
        df_zones.alias("pu"),
        col("trips.PULocationID") == col("pu.LocationID"),
        "left"
    ) \
    .join(
        df_zones.alias("do"),
        col("trips.DOLocationID") == col("do.LocationID"),
        "left"
    )

In [0]:
df_final = df_enriched.select(
    col("trips.*"),
    col("pu.ZoneName").alias("pickup_zone"),
    col("pu.Latitude").alias("pickup_latitude"),
    col("pu.Longitude").alias("pickup_longitude"),

    col("do.ZoneName").alias("dropoff_zone"),
    col("do.Latitude").alias("dropoff_latitude"),
    col("do.Longitude").alias("dropoff_longitude")
    )

df_final.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("taxicatalog.taxi_project_schema.silver_taxi_trips_enriched")

In [0]:
display(df_final)